# RGCNN Implementation subject-independent

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import numpy as np
import pandas as pd
import glob as gb

from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Subset
from DGCNN_Data_Model import PrecomputedEEGDataset, DGCNNBlock 

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
folder_path = '../SEEDiv_processed/filtered_csv' #filtered_csv'
batch_size = 16
channel = 62
feature_bands = 5
num_classes = 4
num_epochs = 50
lr = 0.002

Using device: cuda


#### Full Training Loop

In [3]:
def accuracy(preds, labels):
    pred_labels = torch.argmax(preds, dim=1)
    return (pred_labels == labels).float().mean().item()

In [4]:
def train_dgcnn(model, train_loader, val_loader):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_acc = 0

        for x_batch, label_dict in train_loader:
            x_batch = x_batch.to(device)              # [B, 4, 62, 5]
            # print("W_star shape:", model.W_star.shape)
            # print("Any NaN in W_star?", torch.isnan(model.W_star).any())
            labels = label_dict['emotion'].to(device) # [B]
            logits = model(x_batch)                     # [B, num_classes]
            loss = loss_fn(logits, labels)
            # torch.nn.utils.clip_grad///_norm_(model.parameters(), max_norm=1.0)            
            acc = accuracy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
# 0----------------------------testing------------------------------------
            # for name, param in model.named_parameters():
            #     if torch.isnan(param).any():
            #         print(f"NaN in weights: {name}")
            # for name, param in model.named_parameters():
            #     if param.grad is not None and torch.isnan(param.grad).any():
            #         print(f"NaN in gradients: {name}")

# 0----------------------------testing------------------------------------            
            optimizer.step()

            total_loss += loss.item()
            total_acc += acc

        avg_loss = total_loss / len(train_loader)
        avg_acc = total_acc / len(train_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}")

        # Optional: evaluate on validation set
        if val_loader is not None:
            val_acc = evaluate_dgcnn(model, val_loader, device)
            print(f"→ Validation Acc: {val_acc:.4f}")
    return model

### LOSO Evaluation Loop

In [5]:
@torch.no_grad()
def evaluate_dgcnn(model, data_loader):
    model.eval()
    total_acc = 0

    for x_batch, label_dict in data_loader:
        x_batch = x_batch.to(device)
        labels = label_dict['emotion'].to(device)

        logits = model(x_batch)
        acc = accuracy(logits, labels)
        total_acc += acc

    return total_acc / len(data_loader)

In [6]:
def get_loso_splits(dataset):

    subject_indices = defaultdict(list)

    # Group sample indices by subject
    for idx in range(len(dataset)):
        _, labels = dataset[idx]
        subject = labels['subject']
        subject_indices[subject].append(idx)

    folds = []
    for test_subject in sorted(subject_indices.keys()):
        test_idx = subject_indices[test_subject]
        train_idx = [i for s, idxs in subject_indices.items() if s != test_subject for i in idxs]
        folds.append((train_idx, test_idx))

    return folds# list of (train_idx, test_idx)

In [7]:
def loso_evaluation(dataset, num_classes=4, device='cuda'):
    folds = get_loso_splits(dataset)
    print(len(folds))
    all_accuracies = []

    
    results = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(folds):
        print(f"\n▶️ Fold {fold_idx+1}/{len(folds)} — Leave Subject {fold_idx+1} Out")

        train_set = Subset(dataset, train_idx)
        test_set = Subset(dataset, test_idx)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_set, batch_size=batch_size)

        model = DGCNNBlock(in_features=5, hidden_features=32, num_classes=num_classes, K=4, num_nodes=62)
        model = train_dgcnn(model, train_loader, val_loader=None)

        acc = evaluate_dgcnn(model, test_loader)
        all_accuracies.append(acc)

        print(f" Subject {fold_idx+1} Accuracy: {acc:.4f}")

        results.append({
            "fold": fold_idx + 1,
            "held_out_subject": fold_idx+1,
            "test_accuracy": acc,
            "num_epochs": num_epochs,
            "hidden_dim": 32,
            "cheb_K": 4,
            "lr": lr
        })

    mean_acc = sum(all_accuracies) / len(all_accuracies)
    print(f"\n LOSO Mean Accuracy: {mean_acc:.4f}")
    return results

In [8]:
folder_path = "../SEEDiv_processed/precomputed"
ouput_path = "results"
os.makedirs(ouput_path, exist_ok=True)
features_list = [file for file in sorted(gb.glob(os.path.join(folder_path,'*'))) if file.endswith('.pt')]
labels_list =  [file for file in sorted(gb.glob(os.path.join(folder_path,'*'))) if file.endswith('.csv')]

dataset = PrecomputedEEGDataset(features_list,labels_list)
results = loso_evaluation(dataset, num_classes=4, device='cuda')
df = pd.DataFrame(results)
df.to_csv(os.path.join(ouput_path,"loso_dgcnn_results.csv"), index=False)

15

▶️ Fold 1/15 — Leave Subject 1 Out
Epoch 1/50 | Train Loss: 1.3432 | Train Acc: 0.3343
Epoch 2/50 | Train Loss: 1.2795 | Train Acc: 0.3988
Epoch 3/50 | Train Loss: 1.2508 | Train Acc: 0.4218
Epoch 4/50 | Train Loss: 1.2303 | Train Acc: 0.4353
Epoch 5/50 | Train Loss: 1.2113 | Train Acc: 0.4549
Epoch 6/50 | Train Loss: 1.1955 | Train Acc: 0.4617
Epoch 7/50 | Train Loss: 1.1785 | Train Acc: 0.4735
Epoch 8/50 | Train Loss: 1.1632 | Train Acc: 0.4806
Epoch 9/50 | Train Loss: 1.1517 | Train Acc: 0.4907
Epoch 10/50 | Train Loss: 1.1374 | Train Acc: 0.4981
Epoch 11/50 | Train Loss: 1.1285 | Train Acc: 0.5073
Epoch 12/50 | Train Loss: 1.1151 | Train Acc: 0.5128
Epoch 13/50 | Train Loss: 1.1040 | Train Acc: 0.5193
Epoch 14/50 | Train Loss: 1.0933 | Train Acc: 0.5248
Epoch 15/50 | Train Loss: 1.0801 | Train Acc: 0.5330
Epoch 16/50 | Train Loss: 1.0747 | Train Acc: 0.5362
Epoch 17/50 | Train Loss: 1.0616 | Train Acc: 0.5434
Epoch 18/50 | Train Loss: 1.0539 | Train Acc: 0.5464
Epoch 19/50 | Tr

KeyboardInterrupt: 